In [ ]:
!pip install numpy>=2.2.0 pandas>=2.2.3 scikit-learn>=1.6.0 xgboost>=2.1.0 mplfinance>=0.12.10b0 -q
print("TUDO INSTALADO")

TUDO INSTALADO


## FIT1000

In [ ]:
from datetime import datetime, timezone
import os
import time
import warnings

import numpy as np
import pandas as pd
import requests
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from google.colab import drive

# Monta o Google Drive
drive.mount('/content/drive')

warnings.filterwarnings('ignore')

# =========================================
# CONFIGURAÇÕES GERAIS
# =========================================

ATIVO = 'BTC-USDT'

# O bot principal se mantém buscando sinais
CONFIANCA_MINIMA = 60.0

# O Meta-Modelo tem um sarrafo elevado para aprovar sinais
CONFIANCA_META_MINIMA = 55.0

# Qtd necessária para o bot ligar o filtro avançado de Machine Learning
SINAIS_NECESSARIOS_META = 1000

ARQUIVO_HISTORICO = '/content/drive/MyDrive/historico_sinais_real1000.csv'

# Dias da semana para TREINO: 0=Seg a 6=Dom
DIAS_DA_SEMANA_ANALISE = [0, 1, 2, 3, 4, 5, 6]

# Minutos da análise (sempre 1 min antes da nova vela abrir)
MINUTOS_ENTRADA = ['04', '09', '14', '19', '24', '29', '34', '39', '44', '49', '54', '59']

ultimo_horario_processado = None
sinais_pendentes = []

# Criar cabeçalho do CSV
if not os.path.exists(ARQUIVO_HISTORICO):
    df_init = pd.DataFrame(columns=[
        'timestamp', 'horario_sinal', 'target_ts_ms', 'preco_entrada_sinal',
        'direcao', 'prob_base', 'rsi', 'adx', 'atr', 'dist_ema9',
        'dist_ema21', 'tendencia_m15', 'vol_pct', 'dist_bb_sup', 'dist_bb_inf', 'win_loss'
    ])
    df_init.to_csv(ARQUIVO_HISTORICO, index=False)


# =========================================
# BUSCA DE DADOS
# =========================================

def baixar_candles_completos(timeframe='5m', total_candles=1000):
    url = 'https://www.okx.com/api/v5/market/candles'
    todos_candles = []
    after = ''

    while len(todos_candles) < total_candles:
        params = {'instId': ATIVO, 'bar': timeframe, 'limit': '100'}
        if after:
            params['after'] = after

        try:
            res = requests.get(url, params=params).json()
            dados = res.get('data', [])
            if not dados:
                break

            for c in dados:
                todos_candles.append(c)

            after = dados[-1][0]
        except Exception:
            break

    todos_candles = todos_candles[:total_candles]
    df = pd.DataFrame(todos_candles)

    if df.empty:
        return df

    df.columns = ['time', 'open', 'high', 'low', 'close', 'volume', 'vol_usdt', 'vol_quote', 'confirm']
    df = df[::-1].reset_index(drop=True)

    df['time'] = df['time'].astype(int)
    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = df[col].astype(float)

    return df


# =========================================
# CÁLCULO DE FEATURES
# =========================================

def calcular_features(df_5m):
    df = df_5m.copy()

    df['corpo_pct'] = (df['close'] - df['open']) / df['open']

    amplitude = df['high'] - df['low'] + 1e-8
    df['amplitude_pct'] = amplitude / df['open']
    df['pavio_sup_pct'] = (df['high'] - df[['open', 'close']].max(axis=1)) / amplitude
    df['pavio_inf_pct'] = (df[['open', 'close']].min(axis=1) - df['low']) / amplitude

    ema9 = df['close'].ewm(span=9).mean()
    ema21 = df['close'].ewm(span=21).mean()
    df['dist_ema9'] = (df['close'] - ema9) / df['close']
    df['dist_ema21'] = (df['close'] - ema21) / df['close']

    delta = df['close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    rs = gain.rolling(14).mean() / (loss.rolling(14).mean() + 1e-8)
    df['rsi'] = 100 - (100 / (1 + rs))

    tr = np.maximum(
        df['high'] - df['low'],
        np.maximum(abs(df['high'] - df['close'].shift()), abs(df['low'] - df['close'].shift()))
    )
    df['atr'] = tr.rolling(14).mean() / df['close']
    df['adx'] = (tr.rolling(14).mean() / (df['high'].rolling(14).max() - df['low'].rolling(14).min() + 1e-8)) * 100

    # Volume Relativo
    df['vol_pct'] = df['volume'] / (df['volume'].rolling(20).mean() + 1e-8)

    # Bandas de Bollinger (Distância do topo e do fundo)
    sma20 = df['close'].rolling(20).mean()
    std20 = df['close'].rolling(20).std()
    df['bb_sup'] = sma20 + (2 * std20)
    df['bb_inf'] = sma20 - (2 * std20)
    df['dist_bb_sup'] = (df['bb_sup'] - df['close']) / (df['close'] + 1e-8)
    df['dist_bb_inf'] = (df['close'] - df['bb_inf']) / (df['close'] + 1e-8)

    ema27 = df['close'].ewm(span=27).mean()
    ema63 = df['close'].ewm(span=63).mean()
    df['tendencia_m15'] = (ema27 > ema63).astype(int)

    df['target'] = (df['close'].shift(-1) > df['open'].shift(-1)).astype(int)

    # Removendo NaNs garantindo que as novas métricas de 20 períodos se alinhem
    df = df.dropna(subset=['rsi', 'atr', 'adx', 'dist_ema21', 'tendencia_m15', 'vol_pct', 'dist_bb_sup'])
    return df


# =========================================
# AUDITORIA E META-MODELO
# =========================================

def auditar_sinais_pendentes(df_5m):
    global sinais_pendentes
    sinais_mantidos = []

    for sinal in sinais_pendentes:
        target_ts = sinal['target_ts_ms']
        vela_operada = df_5m[df_5m['time'] == target_ts]

        if not vela_operada.empty and vela_operada.iloc[0]['confirm'] == '1':
            row = vela_operada.iloc[0]
            open_real = row['open']
            close_real = row['close']

            direcao = sinal['direcao']
            if direcao == 'CALL':
                win = 1 if close_real > open_real else 0
            else:
                win = 1 if close_real < open_real else 0

            sinal['win_loss'] = win

            df_hist = pd.read_csv(ARQUIVO_HISTORICO)
            df_novo = pd.DataFrame([sinal])
            df_hist = pd.concat([df_hist, df_novo], ignore_index=True)
            df_hist.to_csv(ARQUIVO_HISTORICO, index=False)

            res_str = '🟢 GREEN (WIN)' if win == 1 else '🔴 RED (LOSS)'
            print(f'\n[AUDITORIA] Sinal de {sinal["horario_sinal"]} ({direcao}) | Open: {open_real:.2f} -> Close: {close_real:.2f} | {res_str}')
        else:
            sinais_mantidos.append(sinal)

    sinais_pendentes = sinais_mantidos


def treinar_meta_modelo():
    if not os.path.exists(ARQUIVO_HISTORICO): return None
    df_hist = pd.read_csv(ARQUIVO_HISTORICO)

    # Bot agora exige a meta profissional de 1000 sinais
    if len(df_hist) < SINAIS_NECESSARIOS_META:
        return None

    features_meta = ['prob_base', 'rsi', 'adx', 'atr', 'dist_ema9', 'dist_ema21', 'tendencia_m15', 'vol_pct', 'dist_bb_sup', 'dist_bb_inf']
    X_meta = df_hist[features_meta]
    y_meta = df_hist['win_loss']

    if len(y_meta.unique()) < 2: return None

    # O max_depth=4 é mantido, ideal para extrair padrões complexos de 1000 registros sem causar overfitting.
    meta_modelo = XGBClassifier(
        n_estimators=100, learning_rate=0.03, max_depth=4,
        random_state=42, eval_metric='logloss'
    )
    meta_modelo.fit(X_meta, y_meta)
    return meta_modelo


# =========================================
# LOOP PRINCIPAL
# =========================================

while True:
    try:
        agora = datetime.now()
        minuto_atual = agora.strftime('%M')
        horario_atual = agora.strftime('%H:%M')

        if minuto_atual not in MINUTOS_ENTRADA:
            time.sleep(10)
            continue

        if horario_atual == ultimo_horario_processado:
            time.sleep(5)
            continue

        ultimo_horario_processado = horario_atual

        print('\n========================================')
        print(f'ANALISANDO BTC EM {horario_atual}')
        print('========================================')

        df_5m = baixar_candles_completos('5m', 1000)

        auditar_sinais_pendentes(df_5m)

        df = calcular_features(df_5m)

        features = [
            'corpo_pct', 'amplitude_pct', 'pavio_sup_pct', 'pavio_inf_pct',
            'dist_ema9', 'dist_ema21', 'rsi', 'atr', 'adx', 'tendencia_m15',
            'vol_pct', 'dist_bb_sup', 'dist_bb_inf'
        ]

        vela_atual_inferencia = df.iloc[-1:]
        df_treino = df.iloc[:-1].copy()

        df_treino['dia_semana'] = pd.to_datetime(df_treino['time'], unit='ms').dt.weekday
        df_treino = df_treino[df_treino['dia_semana'].isin(DIAS_DA_SEMANA_ANALISE)]

        X = df_treino[features]
        y = df_treino['target']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

        modelo_principal = XGBClassifier(
            n_estimators=150,
            learning_rate=0.02,
            max_depth=5,
            min_child_weight=2,
            gamma=0.2,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=42,
            eval_metric='logloss'
        )
        modelo_principal.fit(X_train, y_train)

        acuracia = accuracy_score(y_test, modelo_principal.predict(X_test))

        probs = modelo_principal.predict_proba(vela_atual_inferencia[features])[0]
        prob_verde, prob_vermelha = probs[1] * 100, probs[0] * 100

        adx_atual = vela_atual_inferencia['adx'].iloc[0]
        atr_atual = vela_atual_inferencia['atr'].iloc[0]
        atr_medio = df_treino['atr'].rolling(50).mean().iloc[-1]
        preco_atual = vela_atual_inferencia['close'].iloc[0]

        print(f'ACURÁCIA TESTE (RECENTE): {acuracia * 100:.2f}%')
        print(f'PROB CALL: {prob_verde:.2f}% | PROB PUT: {prob_vermelha:.2f}%')
        print(f'ADX: {adx_atual:.2f} | ATR: {atr_atual:.4f}')

        mercado_operavel = (adx_atual > 15) and (atr_atual < atr_medio * 2.0)

        direcao_sinal = None
        prob_sinal = 0.0

        if prob_verde >= CONFIANCA_MINIMA and mercado_operavel:
            direcao_sinal = 'CALL'
            prob_sinal = prob_verde
        elif prob_vermelha >= CONFIANCA_MINIMA and mercado_operavel:
            direcao_sinal = 'PUT'
            prob_sinal = prob_vermelha

        if direcao_sinal is not None:
            last_ts = vela_atual_inferencia['time'].iloc[0]
            target_ts_ms = last_ts + (5 * 60 * 1000)

            dados_sinal_atual = {
                'timestamp': agora.strftime('%Y-%m-%d %H:%M:%S'),
                'horario_sinal': horario_atual,
                'target_ts_ms': target_ts_ms,
                'preco_entrada_sinal': preco_atual,
                'direcao': direcao_sinal,
                'prob_base': prob_sinal,
                'rsi': vela_atual_inferencia['rsi'].iloc[0],
                'adx': adx_atual,
                'atr': atr_atual,
                'dist_ema9': vela_atual_inferencia['dist_ema9'].iloc[0],
                'dist_ema21': vela_atual_inferencia['dist_ema21'].iloc[0],
                'tendencia_m15': vela_atual_inferencia['tendencia_m15'].iloc[0],
                'vol_pct': vela_atual_inferencia['vol_pct'].iloc[0],
                'dist_bb_sup': vela_atual_inferencia['dist_bb_sup'].iloc[0],
                'dist_bb_inf': vela_atual_inferencia['dist_bb_inf'].iloc[0],
            }

            meta_modelo = treinar_meta_modelo()
            aprovado_pelo_meta = True

            if meta_modelo is not None:
                features_meta = ['prob_base', 'rsi', 'adx', 'atr', 'dist_ema9', 'dist_ema21', 'tendencia_m15', 'vol_pct', 'dist_bb_sup', 'dist_bb_inf']
                X_meta_in = pd.DataFrame([dados_sinal_atual])[features_meta]
                prob_win_meta = meta_modelo.predict_proba(X_meta_in)[0][1] * 100

                print(f'CHANCE DE WIN META-MODELO: {prob_win_meta:.2f}%')

                if prob_win_meta < CONFIANCA_META_MINIMA:
                    aprovado_pelo_meta = False
                    print('\n❌ SINAL CANCELADO (O Meta-Modelo Profissional identificou alto risco de LOSS na database atual)')
            else:
                try:
                    qtd_sinais = len(pd.read_csv(ARQUIVO_HISTORICO))
                except:
                    qtd_sinais = 0
                print(f'META-MODELO EM TREINAMENTO: Coletando dados... ({qtd_sinais}/{SINAIS_NECESSARIOS_META} sinais necessários)')

            if aprovado_pelo_meta:
                print('\n========================================')
                print(f'>>> ENTRADA {direcao_sinal} CONFIRMADA <<<')
                print(f'>>> CONFIANÇA BASE: {prob_sinal:.2f}% <<<')
                print('========================================')
                sinais_pendentes.append(dados_sinal_atual)

        else:
            print('\n========================================')
            print('>>> SEM ENTRADA (CONFIANÇA BAIXA OU MERCADO RUIM) <<<')
            print('========================================')

        time.sleep(60)

    except Exception as e:
        print(f'ERRO NO PROCESSAMENTO: {e}')
        time.sleep(10)